# W09 — Assignment único semanal (Limpieza avanzada + Quality Gates)

## Setup

In [2]:
from pathlib import Path
import duckdb
import os

PROJECT_ROOT = Path(r"C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD").resolve()
os.chdir(PROJECT_ROOT)

DB_PATH = PROJECT_ROOT / "data" / "exoplanets_w09.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DOCS_DIR = PROJECT_ROOT / "docs"
ART_DIR = PROJECT_ROOT / "artifacts"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'","''") + "'"

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

┌───────┐
│ n_raw │
│ int64 │
├───────┤
│  6291 │
└───────┘



## Parte A — Limpieza avanzada

In [3]:
con.execute("DROP TABLE IF EXISTS method_synonyms")

con.execute("""
CREATE TABLE method_synonyms (
  raw_norm VARCHAR PRIMARY KEY,
  canonical VARCHAR NOT NULL
)
""")

con.execute("""
INSERT INTO method_synonyms VALUES
  ('transit', 'transit'),
  ('radial velocity', 'radial_velocity'),
  ('microlensing', 'microlensing'),
  ('imaging', 'imaging'),
  ('transit timing variations', 'transit_timing_variations'),
  ('eclipse timing variations', 'eclipse_timing_variations'),
  ('orbital brightness modulation', 'orbital_brightness_modulation'),
  ('pulsar timing', 'pulsar_timing')
""")

con.sql("SELECT * FROM method_synonyms ORDER BY raw_norm").show()

┌───────────────────────────────┬───────────────────────────────┐
│           raw_norm            │           canonical           │
│            varchar            │            varchar            │
├───────────────────────────────┼───────────────────────────────┤
│ eclipse timing variations     │ eclipse_timing_variations     │
│ imaging                       │ imaging                       │
│ microlensing                  │ microlensing                  │
│ orbital brightness modulation │ orbital_brightness_modulation │
│ pulsar timing                 │ pulsar_timing                 │
│ radial velocity               │ radial_velocity               │
│ transit                       │ transit                       │
│ transit timing variations     │ transit_timing_variations     │
└───────────────────────────────┴───────────────────────────────┘



In [4]:
con.execute("DROP TABLE IF EXISTS silver_planet_v3")

con.execute("""
CREATE TABLE silver_planet_v3 AS
WITH cleaned AS (
  SELECT
    pl_name,
    hostname,
    LOWER(TRIM(hostname)) AS hostname_canon,

    discoverymethod,
    LOWER(TRIM(discoverymethod)) AS discoverymethod_norm,

    TRY_CAST(disc_year AS INTEGER) AS disc_year_int,

    CASE
      WHEN TRY_CAST(disc_year AS INTEGER) IS NULL THEN TRUE
      WHEN TRY_CAST(disc_year AS INTEGER) < 1980 THEN TRUE
      WHEN TRY_CAST(disc_year AS INTEGER) > 2026 THEN TRUE
      ELSE FALSE
    END AS disc_year_bad,

    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    sy_dist,
    ra,
    dec,
    st_teff,
    st_rad,
    st_mass
  FROM raw_ps
  WHERE pl_name IS NOT NULL
    AND hostname IS NOT NULL
)
SELECT
  c.pl_name,
  c.hostname,
  c.hostname_canon,
  c.discoverymethod,
  COALESCE(s.canonical, c.discoverymethod_norm) AS discoverymethod_canon,
  c.disc_year_int,
  c.disc_year_bad,
  c.pl_orbper,
  c.pl_rade,
  c.pl_bmasse,
  c.pl_eqt,
  c.sy_dist,
  c.ra,
  c.dec,
  c.st_teff,
  c.st_rad,
  c.st_mass
FROM cleaned c
LEFT JOIN method_synonyms s
  ON c.discoverymethod_norm = s.raw_norm
WHERE (c.pl_rade IS NULL OR c.pl_rade > 0)
  AND (c.pl_bmasse IS NULL OR c.pl_bmasse > 0)
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v3").show()

con.sql("""
SELECT COUNT(*) AS disc_year_bad
FROM silver_planet_v3
WHERE disc_year_bad
""").show()

con.sql("""
SELECT discoverymethod_canon, COUNT(*) AS n
FROM silver_planet_v3
WHERE discoverymethod_canon IS NOT NULL
GROUP BY discoverymethod_canon
ORDER BY n DESC
LIMIT 15
""").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6291 │
└────────┘

┌───────────────┐
│ disc_year_bad │
│     int64     │
├───────────────┤
│             1 │
└───────────────┘

┌───────────────────────────────┬───────┐
│     discoverymethod_canon     │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ transit                       │  4651 │
│ radial_velocity               │  1181 │
│ microlensing                  │   278 │
│ imaging                       │    97 │
│ transit_timing_variations     │    41 │
│ eclipse_timing_variations     │    17 │
│ orbital_brightness_modulation │     9 │
│ pulsar_timing                 │     8 │
│ astrometry                    │     6 │
│ pulsation timing variations   │     2 │
│ disk kinematics               │     1 │
├───────────────────────────────┴───────┤
│ 11 rows                     2 columns │
└───────────────────────────────────────┘



## Parte B — Quality gates

In [5]:
from datetime import datetime, timezone

run_ts = datetime.now(timezone.utc).isoformat()

con.execute("DROP TABLE IF EXISTS quality_events")

con.execute("""
CREATE TABLE quality_events (
  ts_utc VARCHAR,
  check_name VARCHAR,
  status VARCHAR,
  metric_value BIGINT,
  details VARCHAR
)
""")

con.execute(f"""
INSERT INTO quality_events
SELECT
  '{run_ts}' AS ts_utc,
  'null_pl_name' AS check_name,
  CASE WHEN COUNT(*) - COUNT(pl_name) = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
  (COUNT(*) - COUNT(pl_name))::BIGINT AS metric_value,
  'pl_name debe ser no nulo' AS details
FROM silver_planet_v3
""")

con.execute(f"""
INSERT INTO quality_events
SELECT
  '{run_ts}' AS ts_utc,
  'null_hostname_canon' AS check_name,
  CASE WHEN COUNT(*) - COUNT(hostname_canon) = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
  (COUNT(*) - COUNT(hostname_canon))::BIGINT AS metric_value,
  'hostname_canon debe ser no nulo' AS details
FROM silver_planet_v3
""")

con.execute(f"""
INSERT INTO quality_events
SELECT
  '{run_ts}' AS ts_utc,
  'bad_disc_year' AS check_name,
  CASE WHEN SUM(CASE WHEN disc_year_bad THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'WARN' END AS status,
  SUM(CASE WHEN disc_year_bad THEN 1 ELSE 0 END)::BIGINT AS metric_value,
  'disc_year_int debe estar entre 1980 y 2026 cuando exista' AS details
FROM silver_planet_v3
""")

con.execute(f"""
INSERT INTO quality_events
SELECT
  '{run_ts}' AS ts_utc,
  'invalid_physical_values' AS check_name,
  CASE
    WHEN SUM(
      CASE
        WHEN (pl_orbper IS NOT NULL AND pl_orbper <= 0)
          OR (pl_rade IS NOT NULL AND pl_rade <= 0)
          OR (pl_bmasse IS NOT NULL AND pl_bmasse <= 0)
        THEN 1 ELSE 0
      END
    ) = 0 THEN 'PASS'
    ELSE 'FAIL'
  END AS status,
  SUM(
    CASE
      WHEN (pl_orbper IS NOT NULL AND pl_orbper <= 0)
        OR (pl_rade IS NOT NULL AND pl_rade <= 0)
        OR (pl_bmasse IS NOT NULL AND pl_bmasse <= 0)
      THEN 1 ELSE 0
    END
  )::BIGINT AS metric_value,
  'pl_orbper, pl_rade y pl_bmasse deben ser positivos cuando existan' AS details
FROM silver_planet_v3
""")

con.sql("""
SELECT check_name, status, metric_value
FROM quality_events
ORDER BY check_name
""").show()

┌─────────────────────────┬─────────┬──────────────┐
│       check_name        │ status  │ metric_value │
│         varchar         │ varchar │    int64     │
├─────────────────────────┼─────────┼──────────────┤
│ bad_disc_year           │ WARN    │            1 │
│ invalid_physical_values │ PASS    │            0 │
│ null_hostname_canon     │ PASS    │            0 │
│ null_pl_name            │ PASS    │            0 │
└─────────────────────────┴─────────┴──────────────┘



## Entregable único semanal (W09)

Entrega:
- `assignments/W09_assignment_student.ipynb` ejecutado
- `docs/w09_report.md` (usar template)
- `docs/w09_quality.md` (usar template)
- 1 entrada en `docs/decisions_log.md` (usar template)